# Query generation

# Self metadata query

## Ingestion (metadata enriched)

### Preparing the Chroma DB collection

In [1]:
%pip install -q langchain==1.0.3 langchain-openai==1.0.1 langchain-community==0.4.1 langchain-chroma==1.0.0 langchain-openrouter chromadb==1.3.0 lxml==5.4.0 html2text==2025.4.15 lark==1.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 987.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.

In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import getpass
import os

# OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

try:
    from google.colab import userdata
except ImportError:
    userdata = None

OPENROUTER_API_KEY = None
if userdata is not None:
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass.getpass("Enter your OPENROUTER_API_KEY: ")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ.setdefault("USER_AGENT", "building-llm-applications/ch08-colab")

embedding_model = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

In [3]:
uk_with_metadata_collection = Chroma(
    collection_name="uk_with_metadata_collection",
    # embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY))
    embedding_function=embedding_model)

uk_with_metadata_collection.reset_collection() #A
#A in case it already exists

### Defining content to be ingested and splitting strategy

In [4]:
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [5]:
html2text_transformer = Html2TextTransformer()

In [6]:
text_splitter = RecursiveCharacterTextSplitter( #A
    chunk_size=1000, chunk_overlap=100
)

In [7]:
def split_docs_into_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #B
    chunks = text_splitter.split_documents(
        text_docs)

    return chunks

In [8]:
# uk_destinations = [
#     ("Cornwall", "Cornwall"), ("North_Cornwall", "Cornwall"),
#     ("South_Cornwall", "Cornwall"), ("West_Cornwall", "Cornwall"),
#     ("Tintagel", "Cornwall"), ("Bodmin", "Cornwall"),
#     ("Wadebridge", "Cornwall"),
#     ("Penzance", "Cornwall"), ("Newquay", "Cornwall"),
#     ("St_Ives", "Cornwall"),
#     ("Port_Isaac", "Cornwall"), ("Looe", "Cornwall"),
#     ("Polperro", "Cornwall"),
#     ("Porthleven", "Cornwall"),
#     ("East_Sussex", "East_Sussex"), ("Brighton", "East_Sussex"),
#     ("Battle", "East_Sussex"), ("Hastings_(England)", "East_Sussex"),
#     ("Rye_(England)", "East_Sussex"), ("Seaford", "East_Sussex"),
#     ("Ashdown_Forest", "East_Sussex")
# ]

uk_destinations = [
    ("Cornwall", "Cornwall"), ("North_Cornwall", "Cornwall"),
    ("Polperro", "Cornwall"),
    ("East_Sussex", "East_Sussex"), ("Brighton", "East_Sussex"),
    ("Ashdown_Forest", "East_Sussex")
]
wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

In [9]:
uk_destination_url_with_metadata = [ #C
    ( f'{wikivoyage_root_url}/{destination}', destination, region)
    for destination, region in uk_destinations]

#A Instantiate a relatively fine-chunk splitting strategy
#B Transform HTML docs into clean text docs
#C Prepare metadata to be imported: Url, UK Destination and UK Region

In [10]:
print(uk_destination_url_with_metadata)

[('https://en.wikivoyage.org/wiki/Cornwall', 'Cornwall', 'Cornwall'), ('https://en.wikivoyage.org/wiki/North_Cornwall', 'North_Cornwall', 'Cornwall'), ('https://en.wikivoyage.org/wiki/Polperro', 'Polperro', 'Cornwall'), ('https://en.wikivoyage.org/wiki/East_Sussex', 'East_Sussex', 'East_Sussex'), ('https://en.wikivoyage.org/wiki/Brighton', 'Brighton', 'East_Sussex'), ('https://en.wikivoyage.org/wiki/Ashdown_Forest', 'Ashdown_Forest', 'East_Sussex')]


### Enriching a document with metadata: updating metadata

In [11]:
import time
from langchain_community.document_loaders import AsyncHtmlLoader

WIKIVOYAGE_HEADERS = {
    "User-Agent": (
        "BuildingLLMApplicationsCh10Bot/1.0 "
        "(https://github.com/vidyabhandary/building-llm-applications) "
        "langchain-community/0.4.1"
    ),
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "en",
    "Accept-Encoding": "gzip",
}

_last_wikimedia_request = 0.0
MIN_REQUEST_INTERVAL = 2.0  # seconds between requests

def load_wikivoyage_html(url):
    global _last_wikimedia_request

    # requests_per_second controls concurrency, not spacing between
    # sequential calls -- so we throttle manually here too.
    elapsed = time.monotonic() - _last_wikimedia_request
    if elapsed < MIN_REQUEST_INTERVAL:
        time.sleep(MIN_REQUEST_INTERVAL - elapsed)

    loader = AsyncHtmlLoader(
        web_path=url,
        header_template=WIKIVOYAGE_HEADERS,
        requests_per_second=1,
        trust_env=True,
        ignore_load_errors=False,
        raise_for_status=True,
        preserve_order=True,
    )

    docs = loader.load()
    _last_wikimedia_request = time.monotonic()

    for doc in docs:
        if "Please respect our robot policy" in doc.page_content:
            raise RuntimeError(
                f"Wikimedia rejected the request for {url}. "
                "Stop this ingestion run; do not index or repeatedly "
                "retry the returned policy message."
            )

    return docs

In [12]:
tintagel_url, tintagel_destination, tintagel_region = uk_destination_url_with_metadata[4]

In [13]:
# tintagel_html_loader =AsyncHtmlLoader(tintagel_url)
# tintagel_docs = tintagel_html_loader.load()

tintagel_docs = load_wikivoyage_html(tintagel_url)


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  9.87it/s]


In [14]:
# tintagel_docs # COMMENT: LangChain loaders create docs which contain metadata

In [15]:
for doc in tintagel_docs:
    doc.metadata['destination'] = tintagel_destination
    doc.metadata['region'] = tintagel_region
    print(doc.metadata)

{'source': 'https://en.wikivoyage.org/wiki/Brighton', 'title': 'Brighton – Travel guide at Wikivoyage', 'language': 'en', 'destination': 'Brighton', 'region': 'East_Sussex'}


### Enriching a document with metadata: creating metadata

In [16]:
tintagel_docs_with_metadata = [
    Document(page_content=d.page_content,
             metadata = {
                 'source': tintagel_url,
                 'destination': tintagel_destination,
                 'region': tintagel_region
             })
    for d in tintagel_docs
]

In [17]:
tintagel_docs_with_metadata # examine the Document

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Brighton', 'destination': 'Brighton', 'region': 'East_Sussex'}, page_content='<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="en" dir="ltr">\n<head>\n<meta charset="UTF-8">\n<title>Brighton – Travel guide at Wikivoyage</title>\n<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-

### Enriching the UK destination documents with metadata: creating metadata

In [18]:
for (url, destination, region) in uk_destination_url_with_metadata:
    # html_loader = AsyncHtmlLoader(url) #A
    # docs =  html_loader.load() #B

    docs = load_wikivoyage_html(url)  #A rate-limited, header-safe loader for one destination

    docs_with_metadata = [
        Document(page_content=d.page_content,
        metadata = {
            'source': url,
            'destination': destination,
            'region': region})
        for d in docs]

    chunks = split_docs_into_chunks(docs_with_metadata)

    print(f'Importing: {destination}')
    uk_with_metadata_collection.add_documents(documents=chunks)
#A Loader for one destination
#B Documents (chunks) related to one destination

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  6.13it/s]


Importing: Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 14.32it/s]


Importing: North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  5.58it/s]


...

Importing: East_Sussex


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 10.74it/s]


Importing: Brighton


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  4.49it/s]


Importing: Ashdown_Forest


## Q & A on a collection enriched with metadata

### Searching the collection with a metadata filter explicitly

In [19]:
question =  "Events or festivals"
metadata_retriever = uk_with_metadata_collection.as_retriever(
    search_kwargs={'k':2, 'filter':{'destination': 'Polperro'}})

result_docs = metadata_retriever.invoke(question)

In [20]:
result_docs

[Document(id='bed34749-d470-4e04-a2f5-fffc70f44a8f', metadata={'destination': 'Polperro', 'source': 'https://en.wikivoyage.org/wiki/Polperro', 'region': 'Cornwall'}, page_content="Jump to content\n\nMain menu\n\nMain menu\n\nmove to sidebar hide\n\nNavigation\n\n  * Main page\n  * Travel destinations\n  * Star articles\n  * What's nearby?\n  * Trip planner\n  * Travel forum\n  * Arrivals lounge\n  * Random page\n\nGet involved\n\n  * Travellers' pub\n  * Recent changes\n  * Community portal\n  * Maintenance panel\n  * Policies\n  * Help\n  * Interlingual lounge\n\nSearch\n\nSearch\n\nAppearance\n\n  * Donate\n  * Create account\n  * Log in\n\nPersonal tools\n\n  * Donate\n  * Create account\n  * Log in\n\n## Contents\n\nmove to sidebar hide\n\n  * Beginning\n\n  * 1 Understand\n\n  * 2 Get in\n\nToggle Get in subsection\n\n    * 2.1 By car\n\n    * 2.2 By bus\n\n    * 2.3 On foot\n\n  * 3 Get around\n\n  * 4 See\n\n  * 5 Do\n\n  * 6 Buy\n\n  * 7 Eat\n\n  * 8 Drink\n\n  * 9 Sleep\n\n  *

In [21]:
# COMMENT: As you can see, only chunks associated with'destination': 'Polperro' have been selected

### Generating the self metadata query with the SelfQueryRetriever

In [22]:
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever #A
from langchain_openai import ChatOpenAI
#A this requires pip install lark

In [23]:
metadata_field_info = [
    AttributeInfo(
        name="destination",
        description="The specific UK destination to be searched",
        type="string",
    ),
    AttributeInfo(
        name="region",
        description="The name of the UK region to be searched",
        type="string",
    )
]

In [24]:
question = "Tell me about events or festivals in the UK town of Polperro"

In [28]:
# llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY)

llm = ChatOpenAI(
    model="openai/gpt-5-nano",  # OpenRouter model slugs are prefixed, e.g. "openai/..."
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    max_tokens=4096,
)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm, uk_with_metadata_collection, question,
    metadata_field_info, verbose=True
)

In [29]:
result_docs = self_query_retriever.invoke(question)

In [30]:
result_docs

[Document(id='f389f06c-5413-457e-ad68-215d4f70b225', metadata={'destination': 'Polperro', 'source': 'https://en.wikivoyage.org/wiki/Polperro', 'region': 'Cornwall'}, page_content='## Get around\n\n[edit]\n\n50°19′52″N 4°31′11″W\n\nMap of Polperro\n\nThe walk into town from the parking lot is not very steep and takes 10\nminutes. If walking is not your thing, there\'s a horse and cart or converted\nmilk float "tram" that will take you there and back for £1.50 (75p one way).\n\n## See\n\n[edit]\n\nThere are a couple of art galleries on the walk from the car park to the\nharbour that may be worth a visit.The coastpath to the nearby town of Looe\nalso makes a pleasant walk on a summer\'s day.\n\n  * 50.331685-4.5173881 Polperro Harbour Heritage Museum, 4 The Warren, PL13 2RB, ☏ +44 1503 272423. 10.30AM-4.30PM. £3 (adult). (updated Jul 2022)\n  * 50.331648-4.5213152 Polperro Model Village, Mill Hill, PL13 2RP, ☏ +44 1503 272378. Miniature representation of the village, model railway and mus

### Generating the self metadata query with a LLM function call

#### Query schema

In [31]:
import datetime
from typing import Literal, Optional, Tuple, List

from pydantic import BaseModel, Field
from langchain_classic.chains.query_constructor.ir import (
    Comparator,
    Comparison,
    Operation,
    Operator,
    StructuredQuery,
)
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

In [32]:
class DestinationSearch(BaseModel):
    """Search over a vector database of tourist destinations."""

    content_search: str = Field(
        "",
        description="""Similarity search query applied
        to tourist destinations.""",
    )
    destination: str = Field(
        ...,
        description="The specific UK destination to be searched.",
    )
    region: str = Field(
        ...,
        description="The name of the UK region to be searched.",
    )

    def pretty_print(self) -> None:
        for field in self.__fields__:
            if getattr(self, field) is not None and getattr(
                self, field) != getattr(
                self.__fields__[field], "default", None
            ):
                print(f"{field}: {getattr(self, field)}")

In [33]:
def build_filter(destination_search: DestinationSearch):
    comparisons = []

    destination = destination_search.destination #A
    region = destination_search.region #A

    if destination and destination != '': #B
        comparisons.append(
            Comparison(
                comparator=Comparator.EQ,
                attribute="destination",
                value=destination,
            )
        )
    if region and region != '': #C
        comparisons.append(
            Comparison(
                comparator=Comparator.EQ,
                attribute="region",
                value=region,
            )
        )

    search_filter = Operation(operator=Operator.AND,
                              arguments=comparisons) #D

    chroma_filter = ChromaTranslator().visit_operation(
        search_filter) #E

    return chroma_filter
#A Get destination and region from the structured query
#B If the destination exists, create an 'equality' operation
#C If the region exists, create an 'equality' operation
#D Create a combined search filter
#E Transform the filter into Chroma format

#### Conversion of user question to structured query including metadata filter

In [34]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system_message = """You are an expert at converting user
questions into vector database queries.
You have access to a database of tourist destinations.
Given a question, return a database query optimized
to retrieve the most relevant results.

If there are acronyms or words you are not familiar with,
do not try to rephrase them."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        ("human", "{question}"),
    ]
)
# llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY)

llm = ChatOpenAI(
    model="openai/gpt-5-nano",  # OpenRouter model slugs are prefixed, e.g. "openai/..."
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    max_tokens=4096,
)

structured_llm = llm.with_structured_output(
    DestinationSearch, method="function_calling")
query_generator = prompt | structured_llm

In [35]:
question = "Tell me about events or festivals in the UK town of Polperro"

structured_query =query_generator.invoke(question)

In [36]:
structured_query

DestinationSearch(content_search='events festivals Polperro', destination='Polperro', region='Cornwall')

In [37]:
search_filter = build_filter(structured_query)

In [38]:
search_filter

{'$and': [{'destination': {'$eq': 'Polperro'}},
  {'region': {'$eq': 'Cornwall'}}]}

In [39]:
search_query = structured_query.content_search

In [40]:
search_query

'events festivals Polperro'

In [41]:
metadata_retriever = uk_with_metadata_collection.as_retriever(
    search_kwargs={'k':3, 'filter': search_filter})

In [42]:
answer = metadata_retriever.invoke(search_query)

In [43]:
print(answer)

[Document(id='f389f06c-5413-457e-ad68-215d4f70b225', metadata={'destination': 'Polperro', 'region': 'Cornwall', 'source': 'https://en.wikivoyage.org/wiki/Polperro'}, page_content='## Get around\n\n[edit]\n\n50°19′52″N 4°31′11″W\n\nMap of Polperro\n\nThe walk into town from the parking lot is not very steep and takes 10\nminutes. If walking is not your thing, there\'s a horse and cart or converted\nmilk float "tram" that will take you there and back for £1.50 (75p one way).\n\n## See\n\n[edit]\n\nThere are a couple of art galleries on the walk from the car park to the\nharbour that may be worth a visit.The coastpath to the nearby town of Looe\nalso makes a pleasant walk on a summer\'s day.\n\n  * 50.331685-4.5173881 Polperro Harbour Heritage Museum, 4 The Warren, PL13 2RB, ☏ +44 1503 272423. 10.30AM-4.30PM. £3 (adult). (updated Jul 2022)\n  * 50.331648-4.5213152 Polperro Model Village, Mill Hill, PL13 2RP, ☏ +44 1503 272378. Miniature representation of the village, model railway and mus

In [44]:
## COMMENT: this is only the retrieval step; you still need to wrap it in a RAG chain

# Generating a structured SQL query

## Connecting to the UkBooking database

In [47]:
from langchain_community.utilities import SQLDatabase
from langchain_community.tools import QuerySQLDataBaseTool
from langchain_classic.chains import create_sql_query_chain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import getpass
import os

In [48]:
import sqlite3
from pathlib import Path

db_path = Path("UkBooking.db")

# Allows rerunning the notebook cleanly
if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(db_path)

conn.executescript(
    Path("CreateUkBooking.sql").read_text()
)

conn.executescript(
    Path("PopulateUkBooking.sql").read_text()
)

conn.commit()
conn.close()

print("UkBooking.db created successfully")

UkBooking.db created successfully


In [49]:
db = SQLDatabase.from_uri("sqlite:///UkBooking.db")
print(db.get_usable_table_names())

['Accommodation', 'AccommodationType', 'Booking', 'Customer', 'Destination', 'Offer']


In [50]:
db.run("SELECT * FROM Offer;")

"[(1, 1, 'Summer Special', 0.15, '2024-06-01', '2024-08-31'), (2, 2, 'Weekend Getaway', 0.1, '2024-09-01', '2024-12-31'), (3, 3, 'Early Bird Discount', 0.2, '2024-05-01', '2024-06-30'), (4, 4, 'Stay 3 Nights, Get 1 Free', 0.25, '2024-01-01', '2024-03-31'), (5, 5, 'Historic Stay Offer', 0.1, '2024-04-01', '2024-06-30'), (6, 6, 'Autumn Discount', 0.15, '2024-09-01', '2024-11-30'), (7, 7, 'Cottage Retreat Offer', 0.12, '2024-07-01', '2024-09-30'), (8, 8, 'City Break Deal', 0.08, '2024-10-01', '2024-12-31'), (9, 9, 'Luxury Villa Offer', 0.18, '2024-05-01', '2024-08-31'), (10, 10, 'Spa & Wellness Package', 0.2, '2024-04-01', '2024-07-31')]"

## Generate SQL queries from natural language

In [51]:
# OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

In [52]:
llm = ChatOpenAI(
    model="openai/gpt-5-nano",  # OpenRouter model slugs are prefixed, e.g. "openai/..."
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    max_tokens=4096,
)

### Generating the SQL query

In [53]:
# llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-4.1")
sql_query_gen_chain = create_sql_query_chain(llm, db)
response = sql_query_gen_chain.invoke(
    {"question":
     "Give me some offers for Cardiff, including the hotel name"})

In [54]:
response

'Question: Give me some offers for Cardiff, including the hotel name\nSQLQuery: SELECT\n    "Accommodation"."Name" AS "Name",\n    "Offer"."OfferDescription" AS "OfferDescription",\n    "Offer"."DiscountRate" AS "DiscountRate",\n    "Offer"."StartDate" AS "StartDate",\n    "Offer"."EndDate" AS "EndDate"\nFROM "Accommodation"\nJOIN "Destination" ON "Accommodation"."DestinationId" = "Destination"."DestinationId"\nJOIN "Offer" ON "Accommodation"."AccommodationId" = "Offer"."AccommodationId"\nWHERE "Destination"."Name" = \'Cardiff\'\nLIMIT 5;\nSQLResult:\n"Name" | "OfferDescription" | "DiscountRate" | "StartDate" | "EndDate"\nCardiff Camping | Early Bird Discount | 0.2 | 2024-05-01 | 2024-06-30\nAnswer: Cardiff Camping has an offer: Early Bird Discount (20% off) valid from 2024-05-01 to 2024-06-30.'

In [ ]:
#db.run(response) # returns error

### Executing the SQL query [NOTE: THIS WILL THROW AN ERROR]

In [ ]:
sql_query_exec_chain = QuerySQLDataBaseTool(db=db)
sql_query_gen_chain = create_sql_query_chain(llm, db)
chain = sql_query_gen_chain | sql_query_exec_chain
chain.invoke({"question": "Give me some offers for Cardiff, including the hotel name"})

/tmp/ipykernel_2905/2637313701.py:1: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  sql_query_exec_chain = QuerySQLDataBaseTool(db=db)


'Error: (sqlite3.OperationalError) near "Question": syntax error\n[SQL: Question: Give me some offers for Cardiff, including the hotel name\nSQLQuery: \nSELECT\n  "Accommodation"."Name" AS "HotelName",\n  "Offer"."OfferDescription" AS "OfferDescription",\n  "Offer"."DiscountRate" AS "DiscountRate",\n  "Offer"."StartDate" AS "StartDate",\n  "Offer"."EndDate" AS "EndDate"\nFROM "Offer"\nJOIN "Accommodation" ON "Offer"."AccommodationId" = "Accommodation"."AccommodationId"\nJOIN "Destination" ON "Accommodation"."DestinationId" = "Destination"."DestinationId"\nWHERE "Destination"."Name" = \'Cardiff\'\nORDER BY "Offer"."StartDate" ASC\nLIMIT 5;\nSQLResult: \nHotelName: Cardiff Camping\nOfferDescription: Early Bird Discount\nDiscountRate: 0.2\nStartDate: 2024-05-01\nEndDate: 2024-06-30\nAnswer: Here is an offer for Cardiff including the hotel name: Cardiff Camping offers an Early Bird Discount of 20% (DiscountRate 0.2) from 2024-05-01 to 2024-06-30.]\n(Background on this error at: https://sql

### Fixing the SQL format

In [55]:
clean_sql_prompt_template = """You are an expert in SQL Lite.
You are asked to fix badly formed SQL Lite queries,
which might contain unneded prefixes or suffixes.
Given the following unclean SQL statement,
transform it to a clean,
executable SQL statement for SQL lite.
Always prefix column names with the table name.
Only return an executable SQL statement which terminates
with a semicolon. Do not return anything else.
Do not include the language name or symbols like ```.

Unclean SQL: {unclean_sql}"""

In [56]:
clean_sql_prompt = ChatPromptTemplate.from_template(
    clean_sql_prompt_template)

In [57]:
clean_sql_chain = clean_sql_prompt | llm

In [58]:
full_sql_gen_chain = sql_query_gen_chain | \
   clean_sql_chain | StrOutputParser()

In [59]:
question = """Give me some offers for Cardiff,
including the accomodation name"""

In [60]:
response = full_sql_gen_chain.invoke({"question": question})

In [61]:
response

'SELECT\n  "Accommodation"."Name" AS "Accommodation_Name",\n  "Offer"."OfferDescription" AS "Offer_Description",\n  "Offer"."DiscountRate" AS "Offer_DiscountRate",\n  "Offer"."StartDate" AS "Offer_StartDate",\n  "Offer"."EndDate" AS "Offer_EndDate"\nFROM "Offer"\nJOIN "Accommodation" ON "Offer"."AccommodationId" = "Accommodation"."AccommodationId"\nJOIN "Destination" ON "Accommodation"."DestinationId" = "Destination"."DestinationId"\nWHERE "Destination"."Name" = \'Cardiff\'\nORDER BY "Offer"."StartDate" ASC\nLIMIT 5;'

In [62]:
### Comment: now SQL is fixed

### Executing the SQL query

In [63]:
sql_query_exec_chain = QuerySQLDataBaseTool(db=db)

/tmp/ipykernel_1794/1638333941.py:1: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  sql_query_exec_chain = QuerySQLDataBaseTool(db=db)


In [64]:
sql_query_gen_and_exec_chain = full_sql_gen_chain \
    | sql_query_exec_chain | StrOutputParser()

In [65]:
response = sql_query_gen_and_exec_chain.invoke(
    {"question":question})

In [66]:
response

"[('Cardiff Camping', 'Early Bird Discount', 0.2, '2024-05-01', '2024-06-30')]"

In [ ]:
## COMMENT: this is only the retrieval step; you still need to wrap it in a RAG chain

# Query router

In [67]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableLambda

## Setting up the data retrievers

### Setting up the vector store retriever

In [68]:
tourist_info_retriever_chain = RunnableLambda(
    lambda x: x['question']) \
       | uk_with_metadata_collection.as_retriever(
           search_kwargs={'k':2})

### Setting up the relational database retriever (Same as sql_query_gen_and_exec_chain above)

In [69]:
uk_accommodation_retriever_chain =  full_sql_gen_chain \
    | sql_query_exec_chain | StrOutputParser()

## Setting up the query router

In [70]:
class RouteQuery(BaseModel):
    """Route a user question to the most relevant datasource."""

    datasource: Literal["tourist_info_store",
        "uk_booking_db"] = Field(
        ...,
        description="""Given a user question,
        route it either to a tourist info vector store
        or a UK accomodation booking relational database.""",
    )

# llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-5-nano")
structured_llm_router = llm.with_structured_output(
    RouteQuery) #A
#A Structured router which uses LLM function calls

### Setting up the question router chain

In [71]:
system = """You are an expert at routing a user question
to a tourist info vector store
or to an UK accommodation booking relational database.
The vector store contains tourist information about UK destinations.
Use the vectorstore for general tourist information questions
on UK destinations.
For questions about accommodation availability or booking,
use the UK Booking database."""
route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

### Testing the router chain

In [72]:
selected_data_source = question_router.invoke(
    {"question": "Have you got any offers in Polperro?"}
)

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteQuery(datasource='uk_booking_db'), input_type=RouteQuery])
  return self.__pydantic_serializer__.to_python(


In [73]:
print(selected_data_source)

datasource='uk_booking_db'


In [74]:
selected_data_source = question_router.invoke(
    {"question": "Where are the best beaches in Cornwall?"}
)

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteQuery(datasource='tourist_info_store'), input_type=RouteQuery])
  return self.__pydantic_serializer__.to_python(


In [75]:
print(selected_data_source)

datasource='tourist_info_store'


### Setting up the retriever chooser

In [76]:
retriever_chains = {
    'tourist_info_store': tourist_info_retriever_chain,
    'uk_booking_db': uk_accommodation_retriever_chain
}

def retriever_chooser(question):
    selected_data_source = question_router.invoke(
        {"question": question})

    return retriever_chains[selected_data_source.datasource]

In [77]:
chosen = retriever_chooser("""Tell me about events
or festivals in the UK town of Polperro""")

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteQuery(datasource='tourist_info_store'), input_type=RouteQuery])
  return self.__pydantic_serializer__.to_python(


In [78]:
print(chosen)

first=RunnableLambda(...) middle=[] last=VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7aa7f94e2900>, search_kwargs={'k': 2})


## Setting up the full RAG chain

In [79]:
from langchain_core.runnables import RunnablePassthrough

In [80]:
rag_prompt_template = """
Given a question and some context, answer the question.
If you get a structured context, like a tuple, try to
infer the meaning of the components:
typically they refer to accommodation offers,
and the number is a percentage (0.2 means 20%).
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template)

def execute_rag_chain(question, chosen_retriever):
    full_rag_chain = (
        {
            "context": {"question": RunnablePassthrough()}
                | chosen_retriever,#A
            "question": RunnablePassthrough(),#B
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )

    return full_rag_chain.invoke(question)

#A The context is returned by the retriver after feeding to it the rewritten query
#B This is the original user question

## Executing the full RAG chain

### Question on accommodation offers

In [81]:
question = """Give me some offers for Cardiff,
including the accommodation name"""

chosen_retriever = retriever_chooser(question)

answer = execute_rag_chain(question, chosen_retriever)

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteQuery(datasource='uk_booking_db'), input_type=RouteQuery])
  return self.__pydantic_serializer__.to_python(


In [82]:
print(answer)

- Cardiff Camping: Early Bird Discount — 20% off, valid 2024-05-01 to 2024-06-30.


### Question on tourist information

In [85]:
question_2 = """Tell me about events or festivals
in the UK town of Cornwall"""

chosen_retriever_2 = retriever_chooser(question_2)

answer2 = execute_rag_chain(question_2, chosen_retriever_2)

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteQuery(datasource='tourist_info_store'), input_type=RouteQuery])
  return self.__pydantic_serializer__.to_python(


In [86]:
print(answer2)

Cornwall (a county in the UK) hosts several notable events and festivals. From the provided information, here are some:

- Cornish Film Festival: held annually in November, around Newquay (North Cornwall).
- Royal Cornwall Show: agricultural show at the beginning of June; a major tourist attraction in Cornwall.
- Mummer’s Day (sometimes called “Darkie Day”): an ancient midwinter celebration in Padstow, on Boxing Day and New Year’s Day. Note: the name refers to past customs that some consider politically incorrect today.
- Obby ’Oss: held annually on May Day (1 May) in Padstow, with large marching bands and traditional music; crowds gather, so go early.
- AberFest: a Celtic cultural festival celebrating all things Cornish and Breton, held biennially at Easter in Cornwall.
- Breizh – Kernow Festival: alternates with AberFest; held in Brandivy and Bignan (Breizh/Bretagne, France) on the alternate years.

Optional note: Some of these festivals are not public holidays across the county.


# Retrieval post processing

## RAG Fusion

### Multiple query Generation (same as for MultiQueryRetriver)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

from typing import List
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

In [ ]:
multi_query_gen_prompt_template = """
You are an AI language model assistant. Your task is
to generate five different versions of the given user
question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the
user question, your goal is to help
the user overcome some of the limitations of the
distance-based similarity search.
Provide these alternative questions separated by newlines.
Original question: {question}
"""

multi_query_gen_prompt = ChatPromptTemplate.from_template(
    multi_query_gen_prompt_template)

In [ ]:
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Parse out a question from each output line."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return list(filter(None, lines))


questions_parser = LineListOutputParser()

In [ ]:
llm = ChatOpenAI(model="gpt-5", openai_api_key=OPENAI_API_KEY)

In [ ]:
multi_query_gen_chain = multi_query_gen_prompt | llm | questions_parser

### Reciprocal Rank Fusion algorithm

In [ ]:
# Based on: https://github.com/Raudaschl/rag-fusion/blob/master/main.py

def reciprocal_rank_fusion(results_groups: #A
                           list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple groups of
        ranked documents and an optional parameter k used in
        the Reciprocal Rank Fusion (RRF) formula """

    indexed_results = {} #B

    for group_id, results_group in enumerate(
        results_groups): #V
        for local_rank, doc in enumerate(results_group):
            indexed_results[(group_id, local_rank)] = doc

    fused_scores = {} #D

    for key, doc in indexed_results.items(): #E
        group_id, local_rank = key

        if key not in fused_scores:
            fused_scores[key] = 0 #F

        doc_current_score = fused_scores[key]
        fused_scores[key] += 1 / (local_rank + k) #G

    reranked_results = [ #H
        (indexed_results[key], score)
        for key, score in sorted(fused_scores.items(),
                                 key=lambda x: x[1], reverse=True)
    ]

    return reranked_results
#A Based on: https://github.com/Raudaschl/rag-fusion/blob/master/main.py
# B Initialize a dictionary to organize results with an index
# C Index the results by (group_id, local_rank)
# D Initialize a dictionary to hold fused scores for each unique document
# E Iterate through the indexed results
# F Initialize an indexed result with a score of 0 if it has not been processed yet
# G calculate the new document score with the RRF formula
# H rerank the results by RRF score

In [ ]:
retriever = uk_with_metadata_collection.as_retriever(
    search_kwargs={'k':3})
top_three_results = RunnableLambda(
    lambda x: x[0:3]) #A

rag_fusion_retrieval_chain =multi_query_gen_chain \
    | retriever.map() | reciprocal_rank_fusion \
    | top_three_results #B

docs = rag_fusion_retrieval_chain.invoke(
    {"question": question}) #C
len(docs)
#A select the top three results
#B Full RAG fusion retrieval chain
#C testing the retrieval_chain_rag_fusion chain

3

### Incorporating Rag Fusion into the RAG Chain

In [ ]:
rag_prompt_template = """
Given a question and some context, answer the question.
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template)

rag_chain = (
    {
        "context": {"question": RunnablePassthrough()} | rag_fusion_retrieval_chain,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the step-back question
#B This is the original user question

In [ ]:
user_question = "Can you give me some tips for a trip to Brighton?"

answer = rag_chain.invoke(user_question)
print(answer)

Here are a few tips based on the provided info:

- When to go: The city really comes to life in spring. May brings two major events—Brighton Festival and the Festival Fringe.
- Summer vibe: Brighton flourishes in summer with lazy days and beautiful sunsets along its 5+ mile (8 km) shingle beach facing the English Channel.
- Trip length: A day trip or a long weekend works well year-round.
- Getting there: Trains are a convenient way in (see “Rail travel in Great Britain” for guidance).
- Work options: If you have a working visa, Brighton is good for seasonal and temporary jobs.
- Local info: Check the Brighton & Hove City Council website for updates and event details.
